In [2]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

#load the cleared data from data_exploration

DATA_PATH = Path("../data/processed/bloodmnist_cleaned.npz")

with np.load(DATA_PATH) as data:
    X_train_raw = data["X_train"]
    y_train = data["y_train"]

    X_val_raw = data["X_val"]
    y_val = data["y_val"]

    X_test_raw = data["X_test"]
    y_test = data["y_test"]

    class_names = data["class_names"]

print("Datasets loaded successfully!")

Datasets loaded successfully!


In [3]:
X_train = X_train_raw.astype(np.float32) / 255.0
X_val = X_val_raw.astype(np.float32) / 255.0
X_test = X_test_raw.astype(np.float32) / 255.0

print("Train:")
print("Shape:", X_train.shape)
print("Data type:", X_train.dtype)
print("Min:", X_train.min())
print("Max:", X_train.max())

print("\nValidation:")
print("Shape:", X_val.shape)
print("Data type:", X_val.dtype)

print("\nTest:")
print("Shape:", X_test.shape)
print("Data type:", X_test.dtype)

Train:
Shape: (11943, 28, 28, 3)
Data type: float32
Min: 0.0
Max: 1.0

Validation:
Shape: (1711, 28, 28, 3)
Data type: float32

Test:
Shape: (3420, 28, 28, 3)
Data type: float32


In [4]:
# mean and standard deviation for each RGB channel
channel_means = X_train.mean(axis=(0, 1, 2))
channel_stds = X_train.std(axis=(0, 1, 2))

for channel, mean, std in zip(
    ["Red", "Green", "Blue"],
    channel_means,
    channel_stds
):
    print(f"{channel}: Mean = {mean:.4f}, Std = {std:.4f}")

Red: Mean = 0.8163, Std = 0.2151
Green: Mean = 0.6686, Std = 0.2415
Blue: Mean = 0.6955, Std = 0.1175


In [5]:
# calculate statistics directly from current training data
mu_rgb = np.mean(X_train, axis=(0, 1, 2), dtype=np.float64)
sigma_rgb = np.std(X_train, axis=(0, 1, 2), dtype=np.float64)

print("Original means:", mu_rgb)
print("Original stds:", sigma_rgb)

# reshape for NumPy broadcasting
mu_rgb = mu_rgb.reshape(1, 1, 1, 3)
sigma_rgb = sigma_rgb.reshape(1, 1, 1, 3)

# standardize datasets
X_train_standardized = (
    X_train.astype(np.float64) - mu_rgb
) / sigma_rgb

X_val_standardized = (
    X_val.astype(np.float64) - mu_rgb
) / sigma_rgb

X_test_standardized = (
    X_test.astype(np.float64) - mu_rgb
) / sigma_rgb

# check results
print("\nMeans after standardization:")
print(X_train_standardized.mean(axis=(0, 1, 2)))

print("\nStds after standardization:")
print(X_train_standardized.std(axis=(0, 1, 2)))

Original means: [0.79440726 0.65973585 0.69622221]
Original stds: [0.21560965 0.24157349 0.1178783 ]

Means after standardization:
[8.32028208e-14 5.71517489e-15 2.29265455e-15]

Stds after standardization:
[1. 1. 1.]


In [6]:
X_train_standardized = X_train_standardized.astype(np.float32)
X_val_standardized = X_val_standardized.astype(np.float32)
X_test_standardized = X_test_standardized.astype(np.float32)

print("Train:", X_train_standardized.dtype)
print("Validation:", X_val_standardized.dtype)
print("Test:", X_test_standardized.dtype)

Train: float32
Validation: float32
Test: float32


In [7]:
X_train_rf = X_train.reshape(X_train.shape[0], -1)
X_val_rf = X_val.reshape(X_val.shape[0], -1)
X_test_rf = X_test.reshape(X_test.shape[0], -1)

print("Train:", X_train_rf.shape)
print("Validation:", X_val_rf.shape)
print("Test:", X_test_rf.shape)

Train: (11943, 2352)
Validation: (1711, 2352)
Test: (3420, 2352)


In [8]:
PROCESSED_DIR = Path("../data/processed")
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

np.savez_compressed(
    PROCESSED_DIR / "bloodmnist_preprocessed.npz",
    X_train=X_train,
    X_val=X_val,
    X_test=X_test,
    X_train_standardized=X_train_standardized,
    X_val_standardized=X_val_standardized,
    X_test_standardized=X_test_standardized,
    y_train=y_train,
    y_val=y_val,
    y_test=y_test,
    class_names=class_names
)

print("Preprocessed datasets saved!")

Preprocessed datasets saved!


In [ ]:
import json
from pathlib import Path

MODEL_DIR = Path("../models")
MODEL_DIR.mkdir(parents=True, exist_ok=True)

preprocessing_config = {
    "image_size": [28, 28],
    "channel_means": mu_rgb.reshape(-1).tolist(),
    "channel_stds": sigma_rgb.reshape(-1).tolist(),
    "class_names": [
        str(name) for name in class_names
    ]
}

config_path = MODEL_DIR / "preprocessing_config.json"

with open(config_path, "w", encoding="utf-8") as file:
    json.dump(preprocessing_config, file, indent=4)

print("Preprocessing configuration saved!")
print(preprocessing_config)

Preprocessing configuration saved!
{'image_size': [28, 28], 'channel_means': [0.7944072564051742, 0.6597358542279267, 0.6962222105357052], 'channel_stds': [0.2156096500680687, 0.24157349178199508, 0.1178783026007178], 'class_names': ['basophil', 'eosinophil', 'erythroblast', 'immature granulocytes(myelocytes, metamyelocytes and promyelocytes)', 'lymphocyte', 'monocyte', 'neutrophil', 'platelet']}
